# Lab 26-02 · Reviewers, Blind Spots, Ballots and Royalties

Companion to [26-02](../../chapters/26-02-design-reviews-and-standards.md). Write each prediction down before running its cell.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import reviews as r

## Part 1 · Blind spots

*Predict:* with the average per-reviewer rate fixed at 0.35, what fraction do five reviewers find when half the defects are invisible?

In [2]:
hs = np.linspace(0, 0.5, 51)
found5 = [r.p_found_mix([(1 - h, 0.35 / (1 - h)), (h, 0.0)], 5) for h in hs]
print(f"h = 0.5: five reviewers find {100 * found5[-1]:.1f} %")
ks = np.arange(1, 11)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(100 * hs, 100 * np.array(found5)); ax[0].set_xlabel("hidden fraction (%)"); ax[0].set_ylabel("found by five reviewers (%)")
for h in (0, 0.3, 0.5):
    ax[1].plot(ks, [100 * r.p_found_mix([(1 - h, 0.35 / (1 - h)), (h, 0.0)], k) for k in ks], marker="o", label=f"hidden {int(100 * h)} %")
ax[1].set_xlabel("reviewers"); ax[1].set_ylabel("found (%)"); ax[1].legend()
plt.show()

h = 0.5: five reviewers find 49.9 %


## Part 2 · Capture–recapture bias

*Predict:* biased high or low when 11 of 36 defects are hard?

In [3]:
e1, e2 = 25 * 0.6 + 11 * 0.1, 25 * 0.75 + 11 * 0.1
em = 25 * 0.6 * 0.75 + 11 * 0.1 * 0.1
print(f"estimate from expected counts: {e1 * e2 / em:.1f} (truth 36)")
sim_eq = r.simulate_two_reviews(36, 0.6, 0.75)
sim_het = r.simulate_two_reviews(36, 0.6, 0.75, hard_fraction=11 / 36, hard_p=0.1)
print(f"Chapman mean, equal findability: {sim_eq.mean():.1f}; with hard defects: {sim_het.mean():.1f}")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(sim_eq, bins=40, alpha=0.6, label="equal findability")
ax.hist(sim_het, bins=40, alpha=0.6, label="30 % hard")
ax.axvline(36, color="k"); ax.set_xlabel("Chapman estimate of total defects"); ax.legend()
plt.show()

estimate from expected counts: 28.1 (truth 36)
Chapman mean, equal findability: 36.0; with hard defects: 28.1


## Part 3 · Blocking a ballot

*Predict:* the largest group in which three Do Not Approve voters block a ballot everyone else approves.

In [4]:
for g in range(8, 16):
    res = r.ieee_ballot(g, g - 3, 3, 0)
    print(g, f"approval {100 * res['approval']:.1f} %", "passes" if res["passes"] else "blocked")

8 approval 62.5 % blocked
9 approval 66.7 % blocked
10 approval 70.0 % blocked
11 approval 72.7 % blocked
12 approval 75.0 % passes
13 approval 76.9 % passes
14 approval 78.6 % passes
15 approval 80.0 % passes


## Part 4 · Draft or freeze

*Predict:* the break-even probability when time to market is worth $10 M a month.

In [5]:
from scipy.optimize import brentq
ps = np.linspace(0, 1, 101)
fig, ax = plt.subplots(figsize=(6, 3.5))
for val in (5, 10):
    net = [r.early_start_value(6, val, p, 25, 6) for p in ps]
    pb = brentq(lambda p: r.early_start_value(6, val, p, 25, 6), 0, 1)
    print(f"${val} M/month: break-even p = {pb:.3f}")
    ax.plot(ps, net, label=f"${val} M per month")
ax.axhline(0, color="k", lw=1); ax.set_xlabel("probability of a late change"); ax.set_ylabel("net value of starting early ($M)"); ax.legend()
plt.show()

$5 M/month: break-even p = 0.545
$10 M/month: break-even p = 0.706
